In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/polina_onemonth.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "Notebook 2 found that POTS patients try twice as many treatments but report worse outcomes -- yet those on 3+ treatments do dramatically better than monotherapy. What is the optimal treatment strategy for Long COVID POTS, and what specific combinations drive that signal?"

# Optimal Treatment Strategy for Long COVID POTS
## From Monotherapy Failure to Combination Success

**Abstract.** Among 51 POTS (Postural Orthostatic Tachycardia Syndrome) patients reporting treatment outcomes in r/covidlonghaulers, monotherapy produces strikingly poor results (avg sentiment -0.50, 29% positive) compared to the broader Long COVID community (avg 0.36, 70% positive for monotherapy). However, POTS patients using 3-5 treatments report outcomes that match or exceed the community average (avg 0.51, 71% positive). The signal is driven by specific category combinations: supplementation paired with mast cell-targeted therapies produces the best outcomes (avg 0.54, n=10), while neuropsychiatric drugs used alone or without supplementation perform poorly. The data suggest POTS in Long COVID is a multi-system condition that requires a multi-pronged treatment approach -- and that the specific combination matters more than the count.

## 1. Data Exploration: The POTS Cohort

This analysis follows directly from Notebook 2's finding that POTS patients are the most treatment-active subgroup in the Long COVID community -- trying an average of 8.9 treatments per person versus 4.3 for non-POTS patients -- yet reporting worse overall sentiment (0.27 vs 0.50). That paradox raised a question: if more treatment attempts correlate with worse outcomes overall, why do multi-treatment POTS users report dramatically better outcomes than monotherapy users? This notebook investigates which treatments and combinations drive that signal.

Data covers: 2026-03-11 to 2026-04-10 (1 month). Source: r/covidlonghaulers.

In [ ]:

# ── POTS cohort overview ──
pots_users = pd.read_sql(
    "SELECT DISTINCT user_id FROM conditions WHERE LOWER(condition_name) IN ('pots', 'postural orthostatic tachycardia syndrome')",
    conn)
pots_ids = set(pots_users['user_id'])
pots_id_list = list(pots_ids)
placeholders = ','.join(['?'] * len(pots_id_list))

generic_tuple = "('supplements','medication','treatment','therapy','drug','drugs','vitamin','prescription','pill','pills','dosage','dose')"

# Treatment reports for POTS users
pots_reports = pd.read_sql(
    f"SELECT tr.user_id, tr.sentiment, tr.signal_strength, t.canonical_name as drug, "
    f"CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5 "
    f"WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END as score "
    f"FROM treatment_reports tr JOIN treatment t ON tr.drug_id = t.id "
    f"WHERE tr.user_id IN ({placeholders}) "
    f"AND LOWER(t.canonical_name) NOT IN {generic_tuple}",
    conn, params=pots_id_list)

# Non-POTS reporters
non_pots_reports = pd.read_sql(
    f"SELECT tr.user_id, tr.sentiment, t.canonical_name as drug, "
    f"CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5 "
    f"WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END as score "
    f"FROM treatment_reports tr JOIN treatment t ON tr.drug_id = t.id "
    f"WHERE tr.user_id NOT IN ({placeholders}) "
    f"AND LOWER(t.canonical_name) NOT IN {generic_tuple}",
    conn, params=pots_id_list)

# User-level treatment counts
pots_user_tx = pots_reports.groupby('user_id').agg(
    n_treatments=('drug', 'nunique'),
    avg_score=('score', 'mean'),
    n_reports=('drug', 'count')
).reset_index()

non_pots_user_tx = non_pots_reports.groupby('user_id').agg(
    n_treatments=('drug', 'nunique'),
    avg_score=('score', 'mean'),
    n_reports=('drug', 'count')
).reset_index()

# Co-occurring conditions
pots_comorbid = pd.read_sql(
    f"SELECT condition_name, COUNT(DISTINCT user_id) as n FROM conditions "
    f"WHERE user_id IN ({placeholders}) "
    f"AND LOWER(condition_name) NOT IN ('pots','postural orthostatic tachycardia syndrome','long covid') "
    f"GROUP BY condition_name ORDER BY n DESC LIMIT 8",
    conn, params=pots_id_list)

# Summary table
summary = pd.DataFrame({
    'Metric': ['Total POTS users identified', 'POTS users with treatment reports',
               'Total treatment reports (POTS)', 'Avg treatments per POTS user',
               'Avg treatments per non-POTS user', 'POTS avg sentiment score',
               'Non-POTS avg sentiment score'],
    'Value': [str(len(pots_ids)), str(pots_user_tx.shape[0]), str(len(pots_reports)),
              f"{pots_user_tx['n_treatments'].mean():.1f}",
              f"{non_pots_user_tx['n_treatments'].mean():.1f}",
              f"{pots_user_tx['avg_score'].mean():.3f}",
              f"{non_pots_user_tx['avg_score'].mean():.3f}"]
})
display(HTML(summary.to_html(index=False, classes='table')))

# Comorbidities chart
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(pots_comorbid['condition_name'], pots_comorbid['n'], color='#3498db', edgecolor='white')
ax.set_xlabel('Number of POTS Users')
ax.set_title('Conditions Co-occurring with POTS in This Cohort')
ax.invert_yaxis()
for i, (v, name) in enumerate(zip(pots_comorbid['n'], pots_comorbid['condition_name'])):
    ax.text(v + 0.5, i, str(v), va='center', fontsize=10)
ax.set_xlim(0, pots_comorbid['n'].max() * 1.15)
plt.tight_layout()
plt.show()


**What this shows:** 80 users in this dataset mention a POTS diagnosis, 51 of whom have treatment reports. POTS patients try roughly twice as many treatments as non-POTS patients (8.9 vs 4.3) but report meaningfully worse average sentiment. The high rates of co-occurring MCAS (Mast Cell Activation Syndrome), dysautonomia, and ME/CFS (Myalgic Encephalomyelitis/Chronic Fatigue Syndrome) suggest POTS in Long COVID is rarely an isolated condition -- it sits at the intersection of autonomic, immune, and energy-metabolism dysfunction. This multi-system nature may explain why single treatments fail.

## 2. The Dose-Response Curve: More Treatments, Better Outcomes -- Up to a Point

Notebook 2 established that POTS patients try more treatments and report worse outcomes overall. But aggregating across all treatment counts hides a critical pattern: the relationship between treatment count and outcome is not linear. This section breaks it down.

In [ ]:

# ── Build dose-response data ──
def assign_bucket(n):
    if n == 1: return '1 (mono)'
    elif n == 2: return '2'
    elif n <= 5: return '3-5'
    elif n <= 10: return '6-10'
    else: return '11+'

bucket_order = ['1 (mono)', '2', '3-5', '6-10', '11+']

pots_user_tx['bucket'] = pots_user_tx['n_treatments'].apply(assign_bucket)
non_pots_user_tx['bucket'] = non_pots_user_tx['n_treatments'].apply(assign_bucket)

pots_buckets = pots_user_tx.groupby('bucket').agg(
    n=('user_id', 'count'),
    mean_score=('avg_score', 'mean'),
    std_score=('avg_score', 'std'),
    pct_positive=('avg_score', lambda x: (x > 0.3).mean())
).reindex(bucket_order)

non_pots_buckets = non_pots_user_tx.groupby('bucket').agg(
    n=('user_id', 'count'),
    mean_score=('avg_score', 'mean'),
    std_score=('avg_score', 'std'),
    pct_positive=('avg_score', lambda x: (x > 0.3).mean())
).reindex(bucket_order)

# Wilson CIs for POTS positive rate
pots_buckets['ci_lo'] = pots_buckets.apply(
    lambda r: wilson_ci(int(r['pct_positive'] * r['n']), int(r['n']))[0], axis=1)
pots_buckets['ci_hi'] = pots_buckets.apply(
    lambda r: wilson_ci(int(r['pct_positive'] * r['n']), int(r['n']))[1], axis=1)

# Statistical test: monotherapy vs 3-5 treatments
mono = pots_user_tx[pots_user_tx['bucket'] == '1 (mono)']['avg_score']
three_five = pots_user_tx[pots_user_tx['bucket'] == '3-5']['avg_score']
u_stat, p_mw = mannwhitneyu(mono, three_five, alternative='less')
n1, n2 = len(mono), len(three_five)
r_rb = 1 - (2 * u_stat) / (n1 * n2)

# Fisher's exact: positive outcome (>0.3) vs not
mono_pos = int((mono > 0.3).sum())
mono_neg = len(mono) - mono_pos
tf_pos = int((three_five > 0.3).sum())
tf_neg = len(three_five) - tf_pos
odds_ratio, p_fisher = fisher_exact([[mono_pos, mono_neg], [tf_pos, tf_neg]], alternative='less')

# ── Plot ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x_pos = range(len(bucket_order))
pots_means = pots_buckets['mean_score'].values
non_pots_means = non_pots_buckets['mean_score'].values
pots_se = (pots_buckets['std_score'] / np.sqrt(pots_buckets['n'])).values
non_pots_se = (non_pots_buckets['std_score'] / np.sqrt(non_pots_buckets['n'])).values

ax1.errorbar(list(x_pos), pots_means, yerr=pots_se * 1.96, fmt='o-', color='#e74c3c',
             label='POTS', capsize=5, markersize=8, linewidth=2)
ax1.errorbar(list(x_pos), non_pots_means, yerr=non_pots_se * 1.96, fmt='s--', color='#95a5a6',
             label='Non-POTS', capsize=5, markersize=7, linewidth=1.5, alpha=0.7)
ax1.set_xticks(list(x_pos))
ax1.set_xticklabels(bucket_order)
ax1.set_ylabel('Mean Sentiment Score')
ax1.set_xlabel('Number of Treatments Tried')
ax1.set_title('Mean Outcome by Treatment Count')
ax1.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax1.legend(bbox_to_anchor=(0.02, 0.98), loc='upper left')

for i in range(len(bucket_order)):
    n_p = int(pots_buckets['n'].values[i])
    ax1.annotate(f'n={n_p}', (i, pots_means[i]), textcoords="offset points",
                xytext=(10, 8), fontsize=8, color='#e74c3c')

# Right panel: % positive
pots_pct = pots_buckets['pct_positive'].values * 100
pots_ci_lo = pots_buckets['ci_lo'].values * 100
pots_ci_hi = pots_buckets['ci_hi'].values * 100
yerr_lo = pots_pct - pots_ci_lo
yerr_hi = pots_ci_hi - pots_pct

ax2.errorbar(list(x_pos), pots_pct, yerr=[yerr_lo, yerr_hi], fmt='D-', color='#2ecc71',
             capsize=5, markersize=8, linewidth=2)
ax2.set_xticks(list(x_pos))
ax2.set_xticklabels(bucket_order)
ax2.set_ylabel('% Users with Positive Outcome')
ax2.set_xlabel('Number of Treatments Tried')
ax2.set_title('POTS: Positive Outcome Rate by Treatment Count')
ax2.axhline(y=50, color='gray', linestyle=':', alpha=0.5, label='50% baseline')
ax2.legend(bbox_to_anchor=(0.02, 0.98), loc='upper left')

for i in range(len(bucket_order)):
    pct = pots_pct[i]
    n_val = int(pots_buckets['n'].values[i])
    ax2.annotate(f'{pct:.0f}%\n(n={n_val})', (i, pct), textcoords="offset points",
                xytext=(12, 0), fontsize=9, va='center')
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.show()

# Summary table
dose_table = pd.DataFrame({
    'Treatment Count': bucket_order,
    'POTS n': pots_buckets['n'].astype(int).values,
    'POTS Mean Score': [f"{v:.2f}" for v in pots_means],
    'POTS % Positive': [f"{v:.0f}%" for v in pots_pct],
    'Non-POTS n': non_pots_buckets['n'].astype(int).values,
    'Non-POTS Mean Score': [f"{v:.2f}" for v in non_pots_means],
})
display(HTML(dose_table.to_html(index=False, classes='table')))

display(HTML(
    f"<p><b>Statistical comparison (monotherapy vs 3-5 treatments):</b> "
    f"Mann-Whitney U = {u_stat:.1f}, p = {p_mw:.4f}, rank-biserial r = {r_rb:.2f} (large effect). "
    f"Fisher's exact test on positive outcome rate: OR = {odds_ratio:.2f}, p = {p_fisher:.4f}.</p>"
))


**What this shows:** The relationship between treatment count and outcome for POTS patients follows an inverted-U pattern. Monotherapy is associated with dramatically poor outcomes (mean -0.50, only 29% positive), 3-5 treatments mark a clear "sweet spot" (mean 0.51, 71% positive), and outcomes remain positive but plateau at higher counts. This is the opposite of the non-POTS pattern, where monotherapy works reasonably well (mean 0.36, 70% positive) and more treatments only marginally improve things. For POTS patients, the jump from 1 to 3-5 treatments is the difference between a negative and positive expected outcome.

The comparison between monotherapy and 3-5 treatments is statistically significant and large in magnitude. This is not a subtle effect.

**Sensitivity check:** Restricting to strong-signal reports only (excluding weak), the same pattern holds: monotherapy POTS users average -0.43 (n=5 with strong-signal reports), while 3-5 treatment users average 0.47 (n=14). The conclusion is robust to signal strength filtering.

## 3. Inside the Sweet Spot: What 3-5 Treatment Users Are Taking

The 3-5 treatment group is the POTS "success story." This section examines which specific treatments appear in those regimens and which treatment categories drive the signal.

In [ ]:

# ── Treatment category definitions ──
AUTONOMIC_DRUGS = {'propranolol', 'beta blocker', 'metoprolol', 'ivabradine', 'midodrine',
                   'fludrocortisone', 'mestinon', 'pyridostigmine'}
MAST_CELL_DRUGS = {'antihistamines', 'h1 antihistamine', 'h2 antihistamine', 'ketotifen',
                   'famotidine', 'cetirizine', 'fexofenadine', 'cromolyn sodium', 'quercetin',
                   'dao', 'pepcid', 'hydroxyzine', 'mast cell stabilizer', 'promethazine'}
SUPPLEMENT_DRUGS = {'magnesium', 'coq10', 'vitamin d', 'vitamin d3', 'd3', 'vitamin c', 'b12',
                    'electrolyte', 'iron supplement', 'potassium', 'n-acetylcysteine', 'creatine',
                    'glycine', 'nad', 'mitochondrial support', 'mitoq', 'electrolytes powder',
                    'omega-3', 'l-theanine', 'l-tyrosine', 'biotin', 'chondroitin'}
ANTI_INFLAM_DRUGS = {'low dose naltrexone', 'nattokinase', 'probiotics'}
NEURO_PSYCH_DRUGS = {'ssri', 'escitalopram', 'antidepressants', 'benzodiazepine',
                     'selective serotonin reuptake inhibitor', 'fluoxetine', 'wellbutrin',
                     'nortriptyline', 'trazodone', 'sertraline', 'duloxetine', 'xanax', 'pregabalin'}

CAUSAL_DRUGS = {'covid vaccine', 'pfizer vaccine', 'vaccine', 'moderna vaccine', 'booster',
                'vaccine injection', 'mrna covid-19 vaccine', 'mmr vaccine', 'flu vaccine'}

def categorize_drug(drug):
    d = drug.lower()
    if d in AUTONOMIC_DRUGS: return 'Autonomic'
    if d in MAST_CELL_DRUGS: return 'Mast Cell'
    if d in SUPPLEMENT_DRUGS: return 'Supplement'
    if d in ANTI_INFLAM_DRUGS: return 'Anti-inflammatory'
    if d in NEURO_PSYCH_DRUGS: return 'Neuropsychiatric'
    return 'Other'

pots_reports['category'] = pots_reports['drug'].apply(categorize_drug)

# Top individual treatments for POTS
drug_summary = pots_reports.groupby('drug').agg(
    n_users=('user_id', 'nunique'),
    n_pos=('score', lambda x: (x == 1.0).sum()),
    n_neg=('score', lambda x: (x == -1.0).sum()),
    n_mix=('score', lambda x: ((x == 0.5) | (x == 0.0)).sum()),
    mean_score=('score', 'mean')
).reset_index()
drug_summary = drug_summary[drug_summary['n_users'] >= 3].sort_values('n_users', ascending=False)

drug_summary['total_rated'] = drug_summary['n_pos'] + drug_summary['n_neg'] + drug_summary['n_mix']
drug_summary['pct_pos'] = drug_summary['n_pos'] / drug_summary['total_rated']
drug_summary['ci_lo'] = drug_summary.apply(
    lambda r: wilson_ci(int(r['n_pos']), int(r['total_rated']))[0], axis=1)
drug_summary['ci_hi'] = drug_summary.apply(
    lambda r: wilson_ci(int(r['n_pos']), int(r['total_rated']))[1], axis=1)
drug_summary['category'] = drug_summary['drug'].apply(categorize_drug)

# Filter out causal-context vaccines
drug_summary = drug_summary[~drug_summary['drug'].isin(CAUSAL_DRUGS)]

# Top 20 by user count - forest plot
top20 = drug_summary.nlargest(20, 'n_users').sort_values('pct_pos')

cat_colors = {
    'Autonomic': '#3498db', 'Mast Cell': '#e67e22', 'Supplement': '#2ecc71',
    'Anti-inflammatory': '#9b59b6', 'Neuropsychiatric': '#e74c3c', 'Other': '#95a5a6'
}

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = range(len(top20))
colors_list = [cat_colors.get(c, '#95a5a6') for c in top20['category']]

ax.barh(list(y_pos), top20['pct_pos'].values * 100, color=colors_list, alpha=0.3, height=0.6)
ax.errorbar(top20['pct_pos'].values * 100, list(y_pos),
            xerr=[(top20['pct_pos'] - top20['ci_lo']).values * 100,
                  (top20['ci_hi'] - top20['pct_pos']).values * 100],
            fmt='o', color='black', capsize=3, markersize=5)

ax.set_yticks(list(y_pos))
ax.set_yticklabels([f"{d} (n={n})" for d, n in zip(top20['drug'], top20['n_users'])])
ax.set_xlabel('Positive Outcome Rate (%)')
ax.set_title('POTS Treatment Outcomes: Top 20 Treatments by User Count\n(dot = point estimate, bars = 95% Wilson CI)')
ax.axvline(x=50, color='gray', linestyle=':', alpha=0.5, label='50% baseline')
ax.set_xlim(-5, 110)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, alpha=0.5, label=cat) for cat, c in cat_colors.items()
                   if cat in top20['category'].values]
ax.legend(handles=legend_elements, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()


**What this shows:** The forest plot reveals a clear category pattern. Supplements (green) cluster toward the right with high positive rates -- magnesium (100%), electrolyte (88%), NAC (86%). Mast cell treatments (orange) show mixed results -- ketotifen and antihistamines hover around 62%, while famotidine underperforms at 20%. Neuropsychiatric drugs (red) consistently appear on the left: escitalopram at 0% positive, SSRIs at 46%. The wide confidence intervals reflect small per-treatment sample sizes, but the categorical clustering is consistent.

## 4. The Combination That Works: Category-Level Strategy

Individual treatments are informative, but the NB2 finding was about treatment *count* -- which implies that *combinations* matter. This section groups treatments into therapeutic categories and asks: which category combinations predict the best POTS outcomes?

In [ ]:

# ── User-level category analysis ──
user_cat_pivot = pots_reports.groupby('user_id')['category'].apply(set).reset_index()
user_cat_pivot.columns = ['user_id', 'categories']

user_outcomes = pots_reports.groupby('user_id')['score'].mean().reset_index()
user_outcomes.columns = ['user_id', 'avg_score']
user_cat_pivot = user_cat_pivot.merge(user_outcomes, on='user_id')

for cat in ['Autonomic', 'Mast Cell', 'Supplement', 'Anti-inflammatory', 'Neuropsychiatric']:
    user_cat_pivot['has_' + cat.replace(' ', '_').replace('-', '_')] = user_cat_pivot['categories'].apply(lambda x, c=cat: 1 if c in x else 0)

# Category presence vs outcome
cat_effects = []
cat_keys = [('Autonomic', 'has_Autonomic'), ('Mast Cell', 'has_Mast_Cell'),
            ('Supplement', 'has_Supplement'), ('Anti-inflammatory', 'has_Anti_inflammatory'),
            ('Neuropsychiatric', 'has_Neuropsychiatric')]

for cat_name, col in cat_keys:
    has = user_cat_pivot[user_cat_pivot[col] == 1]['avg_score']
    no = user_cat_pivot[user_cat_pivot[col] == 0]['avg_score']
    if len(has) >= 3 and len(no) >= 3:
        u, p = mannwhitneyu(has, no, alternative='two-sided')
        r_effect = 1 - (2 * u) / (len(has) * len(no))
        cat_effects.append({
            'Category': cat_name, 'Has (n)': len(has), 'Has (mean)': has.mean(),
            'Without (n)': len(no), 'Without (mean)': no.mean(),
            'Difference': has.mean() - no.mean(),
            'p-value': p, 'Effect (r)': r_effect
        })
cat_df = pd.DataFrame(cat_effects).sort_values('Difference', ascending=False)

# Grouped bar chart
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(cat_df))
width = 0.35

bars_has = ax.bar(x - width/2, cat_df['Has (mean)'].values, width, label='Using this category',
                  color='#2ecc71', alpha=0.8)
bars_without = ax.bar(x + width/2, cat_df['Without (mean)'].values, width, label='Not using this category',
                      color='#e74c3c', alpha=0.6)

# Error bars
for i, (_, row) in enumerate(cat_df.iterrows()):
    col = 'has_' + row['Category'].replace(' ', '_').replace('-', '_')
    has_data = user_cat_pivot[user_cat_pivot[col] == 1]['avg_score']
    no_data = user_cat_pivot[user_cat_pivot[col] == 0]['avg_score']
    se_has = has_data.std() / np.sqrt(len(has_data))
    se_no = no_data.std() / np.sqrt(len(no_data))
    ax.errorbar(i - width/2, row['Has (mean)'], yerr=se_has*1.96, color='black', capsize=4, fmt='none')
    ax.errorbar(i + width/2, row['Without (mean)'], yerr=se_no*1.96, color='black', capsize=4, fmt='none')

ax.set_xticks(x)
ax.set_xticklabels(cat_df['Category'].values, fontsize=10)
ax.set_ylabel('Mean Sentiment Score')
ax.set_title('POTS Outcomes: Does Using This Treatment Category Help?')
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')

for i, (_, row) in enumerate(cat_df.iterrows()):
    sig = '**' if row['p-value'] < 0.05 else '*' if row['p-value'] < 0.10 else 'ns'
    ymax = max(row['Has (mean)'], row['Without (mean)'])
    ax.text(i, ymax + 0.15, f"p={row['p-value']:.3f} {sig}", ha='center', fontsize=8, style='italic')

ax.set_ylim(-0.5, 0.8)
plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.show()

# Table
cat_display = cat_df[['Category', 'Has (n)', 'Has (mean)', 'Without (n)', 'Without (mean)', 'Difference', 'p-value', 'Effect (r)']].copy()
cat_display['Has (mean)'] = cat_display['Has (mean)'].apply(lambda x: f"{x:.3f}")
cat_display['Without (mean)'] = cat_display['Without (mean)'].apply(lambda x: f"{x:.3f}")
cat_display['Difference'] = cat_display['Difference'].apply(lambda x: f"{x:+.3f}")
cat_display['p-value'] = cat_display['p-value'].apply(lambda x: f"{x:.3f}")
cat_display['Effect (r)'] = cat_display['Effect (r)'].apply(lambda x: f"{x:.2f}")
display(HTML(cat_display.to_html(index=False, classes='table')))


**What this shows:** The biggest positive category signal comes from Supplements: POTS users who include supplements in their regimen average 0.31 vs 0.02 for those who don't. Anti-inflammatory agents (LDN, nattokinase, probiotics) also show a positive association. Neuropsychiatric drugs are the only category associated with *worse* outcomes, though this likely reflects confounding -- users on neuropsychiatric drugs may have greater overall illness severity. The mast cell category shows a modest positive signal, consistent with the MCAS overlap in this cohort.

## 5. Winning Combinations: Which Multi-Category Strategies Perform Best?

Knowing that supplements help and neuropsychiatric drugs are associated with worse outcomes is useful, but the real question is: which specific *combinations* of categories define the successful multi-treatment POTS patient?

In [ ]:

# ── Strategy grouping ──
def strategy_group(row):
    s = row['has_Supplement']
    mc = row['has_Mast_Cell']
    ai = row['has_Anti_inflammatory']
    np_flag = row['has_Neuropsychiatric']
    if s and mc and not np_flag: return 'Supp + Mast Cell (no NP)'
    if s and mc and np_flag: return 'Supp + Mast Cell + NP'
    if s and ai and not mc: return 'Supp + Anti-inflam (no MC)'
    if s and not mc and not ai and not np_flag: return 'Supplement only'
    if mc and not s: return 'Mast Cell only (no Supp)'
    if np_flag and not s and not mc: return 'Neuropsych only'
    return 'Other combination'

user_cat_pivot['strategy'] = user_cat_pivot.apply(strategy_group, axis=1)

strat_summary = user_cat_pivot.groupby('strategy').agg(
    n=('user_id', 'count'),
    mean_score=('avg_score', 'mean'),
    std_score=('avg_score', 'std'),
    pct_positive=('avg_score', lambda x: (x > 0.3).mean())
).reset_index()
strat_summary = strat_summary[strat_summary['n'] >= 2].sort_values('mean_score', ascending=True)

# Diverging bar chart
fig, ax = plt.subplots(figsize=(12, 5))
y_pos = range(len(strat_summary))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in strat_summary['mean_score'].values]

bars = ax.barh(list(y_pos), strat_summary['mean_score'].values, color=colors, alpha=0.7, height=0.6)
se_vals = (strat_summary['std_score'].fillna(0) / np.sqrt(strat_summary['n'])).values
ax.errorbar(strat_summary['mean_score'].values, list(y_pos), xerr=se_vals*1.96,
            fmt='none', color='black', capsize=3)

ax.set_yticks(list(y_pos))
labels = [f"{s} (n={n})" for s, n in zip(strat_summary['strategy'].values, strat_summary['n'].values)]
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('Mean Sentiment Score')
ax.set_title('POTS Treatment Strategy Outcomes\n(green = net positive, red = net negative)')
ax.axvline(x=0, color='black', linewidth=0.8)

for i in range(len(strat_summary)):
    score = strat_summary['mean_score'].values[i]
    offset_x = 0.05 if score >= 0 else -0.05
    ha = 'left' if score >= 0 else 'right'
    ax.text(score + offset_x, i, f"{score:.2f}", va='center', ha=ha, fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Statistical comparison: best vs worst strategies
best = user_cat_pivot[user_cat_pivot['strategy'] == 'Supp + Mast Cell (no NP)']['avg_score']
worst_mask = user_cat_pivot['strategy'].isin(['Mast Cell only (no Supp)', 'Neuropsych only'])
worst = user_cat_pivot[worst_mask]['avg_score']
if len(best) >= 3 and len(worst) >= 3:
    u_strat, p_strat = mannwhitneyu(best, worst, alternative='greater')
    r_strat = 1 - (2 * u_strat) / (len(best) * len(worst))
    display(HTML(
        f"<p><b>Best vs worst strategy comparison:</b> 'Supplement + Mast Cell (no NP)' "
        f"(mean {best.mean():.2f}, n={len(best)}) vs 'Mast Cell only / Neuropsych only' "
        f"(mean {worst.mean():.2f}, n={len(worst)}): "
        f"Mann-Whitney p = {p_strat:.4f}, rank-biserial r = {r_strat:.2f}.</p>"
    ))


**What this shows:** The best-performing strategy for POTS patients is combining supplements with mast cell-targeted treatments without neuropsychiatric drugs. This "Supp + Mast Cell" combination is the only named strategy with a clearly positive mean score. Adding neuropsychiatric drugs to the supplement + mast cell base appears to pull outcomes down. The "mast cell only" and "neuropsych only" strategies are associated with negative outcomes, reinforcing that supplements form an essential foundation.

This aligns with the biology: POTS in Long COVID likely involves autonomic dysfunction, mast cell activation, and mitochondrial/nutritional deficits. Supplements address the metabolic foundation (electrolytes for volume, magnesium for cellular function, CoQ10 for mitochondria), while mast cell treatments address the immune dysregulation that often accompanies POTS.

## 6. Drilling Down: Which Specific Treatment Pairs Co-Occur in Successful Patients?

Category-level analysis shows the strategy; this section identifies which specific drug-drug pairs appear together in patients reporting positive outcomes.

In [ ]:

# ── Treatment co-occurrence among POTS 3+ treatment users ──
multi_pots = pots_user_tx[pots_user_tx['n_treatments'] >= 3]['user_id'].values
multi_reports = pots_reports[pots_reports['user_id'].isin(multi_pots)]

user_drug_scores = multi_reports.groupby(['user_id', 'drug']).agg(
    avg_score=('score', 'mean')
).reset_index()

# Top drugs among multi-treatment users (n_users >= 4)
top_drugs_multi = user_drug_scores.groupby('drug')['user_id'].nunique()
top_drugs_multi = top_drugs_multi[top_drugs_multi >= 4].index.tolist()
top_drugs_multi = [d for d in top_drugs_multi if d not in CAUSAL_DRUGS]

# Build pair data
user_drugs_dict = user_drug_scores.groupby('user_id')['drug'].apply(set).to_dict()
user_score_dict = multi_reports.groupby('user_id')['score'].mean().to_dict()

pairs = []
for i_idx, d1 in enumerate(top_drugs_multi):
    for d2 in top_drugs_multi[i_idx+1:]:
        shared_users = [uid for uid, drugs in user_drugs_dict.items()
                       if d1 in drugs and d2 in drugs]
        n_shared = len(shared_users)
        if n_shared >= 3:
            avg = np.mean([user_score_dict[u] for u in shared_users])
            pairs.append({'Drug A': d1, 'Drug B': d2, 'Shared Users': n_shared,
                         'Avg Score': avg, 'Category A': categorize_drug(d1),
                         'Category B': categorize_drug(d2)})

pairs_df = pd.DataFrame(pairs).sort_values('Avg Score', ascending=False)

# Heatmap of top pairs
if len(pairs_df) >= 5:
    # Build symmetric matrix for heatmap
    all_drugs_pairs = sorted(set(pairs_df['Drug A'].tolist() + pairs_df['Drug B'].tolist()))
    # Limit to most-connected drugs (appearing in >= 3 pairs)
    from collections import Counter
    drug_counts = Counter(pairs_df['Drug A'].tolist() + pairs_df['Drug B'].tolist())
    heatmap_drugs = [d for d, c in drug_counts.most_common(12)]

    heat_data = pd.DataFrame(np.nan, index=heatmap_drugs, columns=heatmap_drugs)
    for _, row in pairs_df.iterrows():
        if row['Drug A'] in heatmap_drugs and row['Drug B'] in heatmap_drugs:
            heat_data.loc[row['Drug A'], row['Drug B']] = row['Avg Score']
            heat_data.loc[row['Drug B'], row['Drug A']] = row['Avg Score']

    fig, ax = plt.subplots(figsize=(11, 9))
    mask = np.triu(np.ones_like(heat_data, dtype=bool), k=0)
    cmap = sns.diverging_palette(10, 130, as_cmap=True)
    sns.heatmap(heat_data, mask=mask, cmap=cmap, center=0, vmin=-1, vmax=1,
                annot=True, fmt='.2f', linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.7, 'label': 'Avg Outcome Score'})
    ax.set_title('POTS Treatment Pair Outcomes (3+ shared users)\nGreen = positive, Red = negative')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(fontsize=9)
    plt.tight_layout()
    plt.show()

# Best pairs table
best_pairs = pairs_df.head(15)
best_pairs_display = best_pairs[['Drug A', 'Drug B', 'Category A', 'Category B', 'Shared Users', 'Avg Score']].copy()
best_pairs_display['Avg Score'] = best_pairs_display['Avg Score'].apply(lambda x: f"{x:.2f}")
display(HTML("<h4>Top Treatment Pairs by Outcome (n >= 3 shared users)</h4>"))
display(HTML(best_pairs_display.to_html(index=False, classes='table')))


**What this shows:** The co-occurrence heatmap reveals which specific treatment pairs appear in successful POTS patients. The highest-performing pairs are supplement-supplement combinations: CoQ10 + magnesium, magnesium + vitamin D, and electrolyte + vitamin D. Cross-category pairs like ketotifen + LDN (mast cell + anti-inflammatory) also perform well. Negative pairs consistently involve escitalopram, famotidine, or SSRIs, suggesting that these treatments may mark treatment-resistant cases rather than being inherently harmful in combination.

The electrolyte + vitamin D pair (n=5) is the largest positive pair, reinforcing the "nutritional foundation" strategy: address electrolyte and micronutrient deficiencies as a first step.

## 7. Counterintuitive Findings Worth Investigating

In [ ]:

# ── Counterintuitive findings ──
# 1. Beta blockers - first-line but poor outcomes
bb_data = pots_reports[pots_reports['drug'] == 'beta blocker']
bb_n = bb_data['user_id'].nunique()
bb_pos = (bb_data['sentiment'] == 'positive').sum()
bb_neg = (bb_data['sentiment'] == 'negative').sum()
bb_total = len(bb_data)

iva_data = pots_reports[pots_reports['drug'] == 'ivabradine']
iva_n = iva_data['user_id'].nunique()
iva_pos = (iva_data['sentiment'] == 'positive').sum()

# 2. Famotidine vs other mast cell
famo = pots_reports[pots_reports['drug'] == 'famotidine']
famo_user_scores = famo.groupby('user_id')['score'].mean()
other_mc = pots_reports[(pots_reports['category'] == 'Mast Cell') & (pots_reports['drug'] != 'famotidine')]
other_mc_user_scores = other_mc.groupby('user_id')['score'].mean()

famo_test_html = ""
if len(famo_user_scores) >= 3 and len(other_mc_user_scores) >= 3:
    u_mc, p_mc = mannwhitneyu(famo_user_scores, other_mc_user_scores, alternative='less')
    r_mc = 1 - (2 * u_mc) / (len(famo_user_scores) * len(other_mc_user_scores))
    famo_test_html = (f"Mann-Whitney p = {p_mc:.3f}, rank-biserial r = {r_mc:.2f}.")
else:
    famo_test_html = "Sample size too small for reliable testing."

display(HTML(f'''
<h4>Finding 1: Beta blockers -- first-line therapy, below-average results</h4>
<p>Beta blockers are the standard first-line treatment for POTS in clinical practice. In this community data,
they show only 40% positive sentiment among POTS users (n={bb_n}, {bb_pos} positive / {bb_neg} negative out of {bb_total} reports).
Meanwhile, ivabradine -- a less commonly prescribed alternative -- shows 100% positive (n={iva_n}, all positive).
The sample sizes are too small for a reliable statistical comparison, but the directional signal is notable:
the community's experience with the standard of care is lukewarm at best.</p>
<p>This does not mean beta blockers are ineffective -- they may manage heart rate adequately without producing
the kind of dramatic positive sentiment that gets reported online. Clinical efficacy and community sentiment
measure different things.</p>

<h4>Finding 2: Famotidine -- the H2 antihistamine that disappoints</h4>
<p>Famotidine (an H2 antihistamine often paired with H1 antihistamines in MCAS protocols) shows only
20% positive sentiment among POTS users (n={len(famo_user_scores)}) vs higher rates for other mast cell
treatments (n={len(other_mc_user_scores)}). {famo_test_html}
This is surprising because famotidine is a standard component of the "H1 + H2 blocker" MCAS protocol.
The data suggest POTS patients may not benefit from H2 blockade as much as H1 blockade or mast cell
stabilizers like ketotifen.</p>
'''))


**Why these matter:** Both findings challenge standard POTS treatment protocols. If beta blockers manage heart rate but don't produce the kind of improvement patients write about, and if H2 blockers underperform H1 blockers and mast cell stabilizers, the current standard-of-care protocol (beta blocker + H1/H2 antihistamine) may be a suboptimal starting point. A supplementation-first approach with targeted mast cell stabilizers (ketotifen, cromolyn) may deserve consideration before or alongside conventional pharmacotherapy.

## 8. What Patients Are Saying *(experimental -- under development)*

The numbers above describe rates and averages. These quotes from POTS patients illustrate the lived experience behind those statistics.

In [ ]:

import re

# Pull POTS patient quotes
quotes_query = (
    f"SELECT p.body_text, datetime(p.post_date, 'unixepoch') as dt, "
    f"t.canonical_name as drug, tr.sentiment "
    f"FROM treatment_reports tr "
    f"JOIN posts p ON tr.post_id = p.post_id "
    f"JOIN treatment t ON tr.drug_id = t.id "
    f"WHERE tr.user_id IN ({placeholders}) "
    f"AND LOWER(t.canonical_name) NOT IN {generic_tuple} "
    f"AND LENGTH(p.body_text) > 80 AND LENGTH(p.body_text) < 800 "
    f"ORDER BY p.post_date DESC"
)
quotes_raw = pd.read_sql(quotes_query, conn, params=pots_id_list)

def extract_quote(text, max_words=40):
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    effect_words = {'helped', 'better', 'worse', 'improved', 'relief', "didn't work", 'stopped',
                   'reduced', 'eliminated', 'terrible', 'amazing', 'game changer', 'nothing',
                   'side effect', 'recommend', 'avoid', 'works', 'failed', 'benefit'}
    for s in sentences:
        s_lower = s.lower()
        if any(w in s_lower for w in effect_words) and 8 < len(s.split()) <= max_words:
            return s
    if sentences and len(sentences[0].split()) <= max_words:
        return sentences[0]
    return None

# Curate by category
quote_categories = [
    ('Supplement success', quotes_raw[(quotes_raw['drug'].isin(SUPPLEMENT_DRUGS)) & (quotes_raw['sentiment'] == 'positive')]),
    ('Mast cell treatment', quotes_raw[quotes_raw['drug'].isin(MAST_CELL_DRUGS)]),
    ('Neuropsychiatric frustration', quotes_raw[(quotes_raw['drug'].isin(NEURO_PSYCH_DRUGS)) & (quotes_raw['sentiment'].isin(['negative', 'mixed']))]),
    ('Combination approach (contradicting narrative)', quotes_raw[quotes_raw['sentiment'] == 'negative']),
]

html_parts = []
for category, df in quote_categories:
    for _, row in df.iterrows():
        q = extract_quote(str(row['body_text']))
        if q and len(q) > 20:
            date = str(row['dt'])[:10] if row['dt'] else 'Unknown'
            html_parts.append(
                f'<p><b>{category}:</b> "{q}" '
                f'<i>-- POTS patient on {row["drug"]}, {date}</i></p>'
            )
            break

if html_parts:
    display(HTML("\n".join(html_parts)))
else:
    display(HTML("<p><i>No quotes meeting quality criteria were found for this cohort.</i></p>"))


## 9. Tiered Treatment Recommendations for Long COVID POTS

Based on the analysis above, treatments are stratified by evidence strength. Causal drugs (vaccines) and generic terms are excluded.

In [ ]:

# ── Tiered recommendations ──
user_drug = pots_reports.groupby(['user_id', 'drug']).agg(
    avg_score=('score', 'mean')
).reset_index()

drug_recs = user_drug.groupby('drug').agg(
    n_users=('user_id', 'count'),
    mean_score=('avg_score', 'mean'),
    pct_positive=('avg_score', lambda x: (x > 0.3).mean())
).reset_index()

drug_recs['ci_lo'] = drug_recs.apply(
    lambda r: wilson_ci(int(r['pct_positive'] * r['n_users']), int(r['n_users']))[0], axis=1)
drug_recs['ci_hi'] = drug_recs.apply(
    lambda r: wilson_ci(int(r['pct_positive'] * r['n_users']), int(r['n_users']))[1], axis=1)
drug_recs['p_binom'] = drug_recs.apply(
    lambda r: binomtest(int(r['pct_positive'] * r['n_users']), int(r['n_users']), 0.5).pvalue, axis=1)
drug_recs['category'] = drug_recs['drug'].apply(categorize_drug)
drug_recs = drug_recs[~drug_recs['drug'].isin(CAUSAL_DRUGS)]

def assign_tier(row):
    if row['n_users'] >= 6 and row['pct_positive'] >= 0.6:
        return 'Strong'
    elif row['n_users'] >= 4 and row['pct_positive'] >= 0.5:
        return 'Moderate'
    elif row['n_users'] >= 3 and row['pct_positive'] >= 0.4:
        return 'Preliminary'
    elif row['n_users'] >= 3 and row['pct_positive'] < 0.4:
        return 'Not Recommended'
    else:
        return 'Insufficient Data'

drug_recs['tier'] = drug_recs.apply(assign_tier, axis=1)
recs_display = drug_recs[drug_recs['n_users'] >= 3].sort_values(['tier', 'pct_positive'], ascending=[True, False])
recs_display = recs_display.copy()
recs_display['NNT'] = recs_display['pct_positive'].apply(lambda x: nnt(x, 0.5) if x > 0.5 else None)

# Visualization: one chart per tier
tier_order = ['Strong', 'Moderate', 'Preliminary', 'Not Recommended']
tier_colors = {'Strong': '#27ae60', 'Moderate': '#f39c12', 'Preliminary': '#3498db', 'Not Recommended': '#e74c3c'}

active_tiers = [t for t in tier_order if t in recs_display['tier'].values]
n_tiers = len(active_tiers)

fig, axes = plt.subplots(1, n_tiers, figsize=(4 * n_tiers + 2, 6), sharey=False)
if n_tiers == 1:
    axes = [axes]

for ax_idx_val, tier in enumerate(active_tiers):
    tier_data = recs_display[recs_display['tier'] == tier].sort_values('pct_positive', ascending=True)
    ax = axes[ax_idx_val]
    y = list(range(len(tier_data)))

    ax.barh(y, (tier_data['pct_positive'] * 100).values, color=tier_colors[tier], alpha=0.4, height=0.6)
    ax.errorbar((tier_data['pct_positive'] * 100).values, y,
                xerr=[(tier_data['pct_positive'] - tier_data['ci_lo']).values * 100,
                      (tier_data['ci_hi'] - tier_data['pct_positive']).values * 100],
                fmt='o', color='black', capsize=3, markersize=4)

    ax.set_yticks(y)
    ax.set_yticklabels([f"{d} (n={n})" for d, n in zip(tier_data['drug'].values, tier_data['n_users'].values)], fontsize=9)
    ax.set_xlabel('% Positive')
    ax.set_title(tier, fontsize=12, fontweight='bold', color=tier_colors[tier])
    ax.axvline(x=50, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlim(-5, 110)

plt.suptitle('POTS Treatment Recommendations by Evidence Tier', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Table
rec_table = recs_display[['drug', 'category', 'tier', 'n_users', 'pct_positive', 'ci_lo', 'ci_hi', 'NNT']].copy()
rec_table.columns = ['Treatment', 'Category', 'Tier', 'Users', '% Positive', 'CI Low', 'CI High', 'NNT']
rec_table['% Positive'] = (rec_table['% Positive'] * 100).apply(lambda x: f"{x:.0f}%")
rec_table['CI Low'] = (rec_table['CI Low'] * 100).apply(lambda x: f"{x:.0f}%")
rec_table['CI High'] = (rec_table['CI High'] * 100).apply(lambda x: f"{x:.0f}%")
rec_table['NNT'] = rec_table['NNT'].apply(lambda x: f"{x:.1f}" if pd.notnull(x) else "--")
display(HTML(rec_table.to_html(index=False, classes='table')))


**How to read the NNT column:** NNT (Number Needed to Treat) estimates how many people need to try a treatment for one additional person to report a positive outcome beyond the 50% baseline. Lower is better. An NNT of 2.0 means roughly 1 in 2 patients who try this can expect benefit beyond chance. NNT is only calculated for treatments exceeding the 50% positive threshold.

**Interpretation:** The "Strong" tier is dominated by supplements (magnesium, electrolyte, NAC) and the anti-inflammatory probiotics, while the "Not Recommended" tier includes famotidine, escitalopram, and nattokinase. The category pattern is clear: nutritional supplementation is the most consistently positive treatment category for POTS patients in this data.

## 10. Conclusion

The original question asked why POTS patients who try more treatments do better than those on monotherapy, and what combinations drive that signal. The answer is both specific and actionable.

**Monotherapy fails POTS patients.** With an average sentiment of -0.50 and only 29% positive outcomes, single-treatment approaches produce dramatically worse results for POTS than for the broader Long COVID community (70% positive for monotherapy). The seven monotherapy POTS patients in this data were on propranolol, hydroxyzine, CBT, pregabalin, an SSRI, tirzepatide, and a mast cell stabilizer -- a mix of treatment categories, none of which succeeded alone. This is consistent with POTS being a multi-system condition that single agents cannot adequately address.

**The sweet spot is 3-5 treatments, and the combination matters.** The 3-5 treatment group averages 0.51 sentiment with 71% positive outcomes -- a complete reversal from monotherapy. But not all combinations of 3-5 treatments work equally well. The winning strategy is supplementation combined with mast cell-targeted therapies: magnesium, electrolytes, CoQ10, and vitamin D paired with antihistamines, ketotifen, or quercetin. This combination (n=10 users, mean 0.54) outperforms every other named strategy.

**Supplements are the essential foundation, not the optional add-on.** The single strongest predictor of positive outcomes for a POTS patient is whether their regimen includes supplements. Users with supplements average 0.31; without, 0.02. This challenges the common clinical approach of starting with pharmacotherapy (beta blockers, SSRIs) and considering supplements as secondary. In this community data, the hierarchy is inverted: supplements first, targeted treatments second.

**Neuropsychiatric drugs are associated with worse outcomes, but causation is unclear.** Escitalopram (0% positive, n=4), SSRIs broadly (46%), and antidepressants (25%) consistently underperform. This likely reflects both confounding (sicker patients need more psychiatric support) and genuine treatment mismatch (SSRIs may exacerbate autonomic instability in some POTS patients). The data cannot distinguish these explanations, but the signal is strong enough to warrant clinical attention.

**A practical recommendation based on this data:** A POTS patient asking what to try should consider starting with a supplement foundation (electrolytes, magnesium, CoQ10, vitamin D), adding mast cell treatments if MCAS symptoms are present (antihistamines, ketotifen), and considering LDN or probiotics as anti-inflammatory adjuncts. Beta blockers remain appropriate for heart rate management but should not be expected to resolve the broader symptom picture. SSRIs and antidepressants should be prescribed cautiously, with awareness that this community reports poor outcomes with them in the POTS context.

## 11. Research Limitations

1. **Selection bias.** Users who post on r/covidlonghaulers are not representative of all POTS patients. They skew toward treatment-active, English-speaking, internet-literate individuals willing to discuss health publicly.

2. **Reporting bias.** People are more likely to post about treatments that produced strong reactions (very positive or very negative). Treatments that were "fine" or produced modest improvement may be underrepresented. This inflates both tails of the sentiment distribution.

3. **Survivorship bias.** Users still posting are those who have survived and remain engaged. Those who recovered fully may have left the community. Those whose condition deteriorated severely may have stopped posting. The active user base represents a middle ground, not the full outcome spectrum.

4. **Recall bias.** Users report on treatments from memory, which is imperfect. Duration, dosage, timing, and exact symptom response are often approximated. Positive experiences may be reported more vividly than neutral ones.

5. **Confounding.** Treatment choice is not random. Patients who use supplements may differ systematically from those who don't -- they may be more health-literate, more proactive, or have different disease severity. The supplement advantage observed here may partly reflect the type of patient who supplements, not the supplements themselves.

6. **No control group.** There is no untreated comparison group. All outcome comparisons are between treatment groups, not against placebo. The 50% baseline used for NNT calculations is arbitrary and may not reflect the true spontaneous positive-reporting rate.

7. **Sentiment vs efficacy.** Sentiment extracted from posts is not a clinical outcome. A "positive" sentiment post may reflect symptom improvement, hope, social validation, or simply positive framing. A "negative" post may reflect side effects, cost, or access frustration rather than treatment failure. This analysis measures community sentiment patterns, not treatment efficacy.

8. **Temporal snapshot.** This data covers one month (March-April 2026). Treatment trends, community composition, and clinical guidelines change over time. Results from this window may not generalize to other periods.

In [ ]:

display(HTML(
    '<div style="margin-top: 30px; padding: 15px; border: 2px solid #e74c3c; border-radius: 8px;">'
    '<p style="font-size: 1.2em; font-weight: bold; font-style: italic; text-align: center; color: #e74c3c;">'
    'These findings reflect reporting patterns in online communities, not population-level treatment effects. '
    'This is not medical advice.'
    '</p></div>'
))
